In [ ]:
import jax
import jax.numpy as jnp


def f(x):
    return jnp.concatenate([x, x**2])


x = jnp.array([1.0, 2.0, 3.0])
v = jnp.array([0.1, 0.2, 0.3])

# Compute grad f
jac_f = jax.jacobian(f)

# Compute Hessian-vector product using jax.jvp on grad_f
hvp_fast = jax.jvp(jac_f, (x,), (v,))[1]

hessian = jax.hessian(f)(x)
hvp_exact = hessian @ v

print("Value:", f(x))
print("Jacobian", jac_f(x), jac_f(x).shape)
print("Hessian", hessian.shape)
print("hvp fast", hvp_fast, hvp_fast.shape)
print("hvp exact", hvp_exact, hvp_exact.shape)


Value: [1. 2. 3. 1. 4. 9.]
Jacobian [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]
 [2. 0. 0.]
 [0. 4. 0.]
 [0. 0. 6.]] (6, 3)
Hessian (6, 3, 3)
hvp fast [[0.  0.  0. ]
 [0.  0.  0. ]
 [0.  0.  0. ]
 [0.2 0.  0. ]
 [0.  0.4 0. ]
 [0.  0.  0.6]] (6, 3)
hvp exact [[0.  0.  0. ]
 [0.  0.  0. ]
 [0.  0.  0. ]
 [0.2 0.  0. ]
 [0.  0.4 0. ]
 [0.  0.  0.6]] (6, 3)


In [15]:
import jax
import jax.numpy as jnp


# h(x) as given
def h(x):
    return jnp.concatenate([x, x**2])


# Inputs
x = jnp.array([0.1, 0.2, 0.3])
y = jnp.array([1.0, 2.0, 3.0, 4.0, 5.0, 6.0])


# VJP of h(x) w.r.t. x, pulled back by y
def vjp_h_y(x):
    _, vjp_fun = jax.vjp(h, x)
    return vjp_fun(y)[0]


# Linearize it once at x
primals_out, pullback = jax.linearize(vjp_h_y, x)

# Now HVP is just pullback applied to v
v = jnp.array([0.5, 0.5, 0.5])
hvp = pullback(v)

print("fast hvp:", hvp)


def f(x):
    return jnp.dot(y, h(x))


print("exact hvp:", jax.hessian(f)(x) @ v)


fast hvp: [4. 5. 6.]
exact hvp: [4. 5. 6.]
